# 第 2 章 単回帰: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/02-simple-regression.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0             # 乱数の種
n = 50               # 標本の大きさ
intercept = 1.0      # 真の切片
slope = 0.5          # 真の傾き
sigma = 1.0          # 誤差の標準偏差
repetitions = 1000   # 標本を取り直す回数

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and (
    Path(getattr(_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 1 つの標本で、直線を当てはめる

In [ ]:
import numpy as np

from lib.chapter02.simple import (
    draw_sample,
    fit_simple,
    r_squared,
    sampling_distribution,
    slope_standard_error,
)

rng = np.random.default_rng(seed)
x, y = draw_sample(n, intercept, slope, sigma, rng)
fit = fit_simple(x, y)
print(f"切片 {fit.intercept:.3f}、傾き {fit.slope:.3f}（真の値 {slope}）")
print(f"R² {r_squared(fit, x, y):.3f}、傾きの標準誤差 "
      f"{slope_standard_error(fit, x, y):.3f}")

table = sampling_distribution(n, repetitions, intercept, slope, sigma, rng)
print(f"標本を {repetitions} 回取り直した傾き: 平均 {table['slope'].mean():.3f}、"
      f"標準偏差 {table['slope'].std(ddof=1):.3f}")

## 図: 回帰直線と残差

In [ ]:
from lib.chapter02.plot import plot_regression_line

plot_regression_line(x, y, fit)

## 図: 傾きの推定値の分布

In [ ]:
from lib.chapter02.plot import plot_slope_distribution

plot_slope_distribution(table["slope"].tolist(), slope)

## 試してみよう

- `n` を 200 にすると、傾きの標準誤差は 0.159 から 0.076 に下がり、
  取り直した傾きのばらつきも 0.151 から 0.070 になります（標本が大きいほど、
  推定は安定します）
- `sigma` を 3.0 にすると、R² は 0.002、標準誤差は 0.476 になります。誤差が大きいと、
  傾きは、ほとんど分かりません
- `slope` を 0.0 にしても、1 つの標本の傾きは −0.119 と、0 にはなりません。
  標本を取り直した傾きの平均は −0.003 で、0 の近くに集まります